# MSI Status for samples in cBioPortal with frameshift variants in homopolymer runs

#load all clinical information for samples in cBioPortal and annotate variants in homopolymers with MSI Status

In [ ]:
# 1) import modules
import os
import re
import json
import pandas as pd
import time
import hashlib
import csv
import random
import numpy as np    

In [9]:
cbioall=pd.read_csv("cBio_all_clinical_fromRStudio_6-14-23.txt", sep="\t")

In [33]:
start_time_block2=time.asctime(time.localtime())
print(start_time_block2)
parentpath="/Users/suhasinilulla/Library/CloudStorage/Box-Box/PhD_BCM/Lulla_Plon_Lab/High_priority_gene_list/GL_S_Pipeline_1st_pass_47TSGs_May2023"
os.chdir(parentpath)

#read file with folder names and MANE Select transcript identifiers (or MANE Plus Clinical in the case of SMARCA4)
TranscriptID= pd.read_csv("TSGfolder_MANESelect_names_aalength_GeneIDs_6-13-24.txt", sep="\t")

MSI_df=pd.DataFrame()

for index, row in TranscriptID.iterrows():
    
    # Loop through each folder name
    folder_name=TranscriptID.loc[index]["Gene Symbol"]
    #if folder name exists, list all files in that folder then change directory to that folder
    if os.path.exists(folder_name) and os.path.isdir(folder_name):
        filelist=os.listdir(folder_name)
        os.chdir(folder_name)
           
        # saved file of somatic data within limits of somatic frameshift clusters as 'Python_cbio_processed_VEP_MANE_CAID_CDSLocationfiltered_OTtissue_10-24-24_fsTMB_3-24-25.txt'
        if os.path.exists("Python_coderun_7-11-24_cbio_process_again/Python_cbio_processed_VEP_MANE_CAID_CDSLocationfiltered_OTtissue_10-24-24_fsTMB_3-24-25.txt"):
            cbio_processed_rawdata_fs=pd.read_csv("Python_coderun_7-11-24_cbio_process_again/Python_cbio_processed_VEP_MANE_CAID_CDSLocationfiltered_OTtissue_10-24-24_fsTMB_3-24-25.txt", sep="\t")
            #display(cbio_processed_rawdata_fs["HGVSc"].value_counts())
            cbioformsi=cbioall.set_index("uniqueSampleKey").join(cbio_processed_rawdata_fs.set_index("uniqueSampleKey"), how="inner", rsuffix="_cbiofsfile32425")
            MSI_df=pd.concat([MSI_df,cbioformsi])

            print(len(MSI_df))
        
        print("Current directory:", os.getcwd())
        os.chdir(parentpath)
    else:
        print(f"Folder '{folder_name}' not found.")

end_time_block2=time.asctime(time.localtime())
print(end_time_block2)

Fri Jan  9 13:33:27 2026
Current directory: /Users/suhasinilulla/Library/CloudStorage/Box-Box/PhD_BCM/Lulla_Plon_Lab/High_priority_gene_list/GL_S_Pipeline_1st_pass_47TSGs_May2023/WT1
Current directory: /Users/suhasinilulla/Library/CloudStorage/Box-Box/PhD_BCM/Lulla_Plon_Lab/High_priority_gene_list/GL_S_Pipeline_1st_pass_47TSGs_May2023/VHL
Current directory: /Users/suhasinilulla/Library/CloudStorage/Box-Box/PhD_BCM/Lulla_Plon_Lab/High_priority_gene_list/GL_S_Pipeline_1st_pass_47TSGs_May2023/TSC2
Current directory: /Users/suhasinilulla/Library/CloudStorage/Box-Box/PhD_BCM/Lulla_Plon_Lab/High_priority_gene_list/GL_S_Pipeline_1st_pass_47TSGs_May2023/TSC1
Current directory: /Users/suhasinilulla/Library/CloudStorage/Box-Box/PhD_BCM/Lulla_Plon_Lab/High_priority_gene_list/GL_S_Pipeline_1st_pass_47TSGs_May2023/TP53
113
Current directory: /Users/suhasinilulla/Library/CloudStorage/Box-Box/PhD_BCM/Lulla_Plon_Lab/High_priority_gene_list/GL_S_Pipeline_1st_pass_47TSGs_May2023/SUFU
Current directory: 

In [ ]:
MSI_df_MSIattributes=MSI_df[MSI_df["clinicalAttributeId"].str.contains("MSI_STATUS")==True]

In [ ]:
MSI_df_MSIattributes_uniquesamples=MSI_df_MSIattributes.reset_index().drop_duplicates(subset="uniqueSampleKey")

In [149]:
MSI_df_MSIattributes_uniquesamples["value"].value_counts()

value
high             84
MSS              17
indeterminant     2
MSI-high          1
Name: count, dtype: int64

In [150]:
MSI_df_MSIattributes_uniquesamples["value"].value_counts().sum()

104